# 🦙 Open GenAI Stack과 RAG: 지능형 시스템 만들기

이 노트북에서는 **Open GenAI Stack을 사용한 검색 증강 생성(Retrieval-Augmented Generation, RAG)**을 보여 줍니다. RAG는 AI 모델이 외부 문서와 지식 베이스에 접근하여 이를 바탕으로 추론할 수 있게 해 주는 강력한 기법입니다.

In [ ]:
import uuid

from llama_stack_client import LlamaStackClient
from termcolor import cprint

import sys
sys.path.append('..')

## 🔗 OGX에 연결하기

모든 RAG 작업을 조율하는 AI 엔진인 OGX에 연결합니다. OGX는 다음 작업을 조율하는 중앙 허브 역할을 합니다.
- 벡터 데이터베이스 작업(저장 및 검색)
- 문서 처리와 청크 분할
- 검색된 컨텍스트를 활용한 LLM 추론
- 에이전트 워크플로와 도구 사용

In [ ]:
# base URL은 배포된 Llama Stack 서버를 가리킵니다
base_url = "http://llama-stack-service:8321"

# Llama Stack 클라이언트 생성
client = LlamaStackClient(
    base_url=base_url,
    provider_data=None
)

print(f"Connected to Llama Stack server")

# 모델과 샘플링 설정
model_id = "vllm-llama32/llama32"
temperature = 0.0
max_tokens = 512
stream = False

# temperature에 따라 샘플링 전략 설정
if temperature > 0.0:
    top_p = 0.95  # Nucleus 샘플링 파라미터
    strategy = {"type": "top_p", "temperature": temperature, "top_p": top_p}
else:
    strategy = {"type": "greedy"}  # 항상 가장 확률이 높은 토큰 선택

# 추론 API에 전달할 샘플링 파라미터 묶기
sampling_params = {
    "strategy": strategy,
    "max_tokens": max_tokens,
}

print(f"Model: {model_id}")
print(f"Sampling Parameters: {sampling_params}")
print(f"Stream: {stream}")

## 벡터 데이터베이스 등록하기

In [ ]:
# Llama Stack에 벡터 데이터베이스에 연결하고 사용하는 방법을 알려 줍니다
vs = client.vector_stores.create(
    name="my_citations_db",
    extra_body={
        "embedding_model": "sentence-transformers/nomic-ai/nomic-embed-text-v1.5",
        "embedding_dimension": 768,
        "provider_id": "milvus",
        "vector_db_id": "test"
    }
)

print(f"📊 Created vector database with ID: {vs.id}")

## 📚 문서 수집과 처리

이제 **RAG 레이어**가 활약할 차례입니다! Llama Stack의 RAG Tool을 사용해 다음 작업을 자동으로 수행합니다.

1. URL에서 문서를 **다운로드**합니다
2. PDF 콘텐츠를 **처리**하고 텍스트를 추출합니다
3. 큰 문서를 적절한 크기(각 512 토큰)로 **청크 분할**합니다
4. 임베딩 모델로 각 청크를 **임베딩**합니다
5. 벡터와 메타데이터를 벡터 데이터베이스에 **저장**합니다

**문서를 수집하는 두 가지 방법:**
- **Direct Vector IO**: 미리 처리한 청크를 벡터 데이터베이스에 직접 삽입합니다
- **Llama Stack RAG Tool**(여기서 사용하는 방법): URL이나 파일에서 자동으로 처리합니다

RAG Tool을 사용하면 URL, 파일 등에서 문서를 쉽게 수집하고 자동으로 작은 조각으로 나눌 수 있으므로 이 방법을 사용합니다. 

In [ ]:
# 처리할 URL 목록(여기서는 1개)
urls = [
    "https://raw.githubusercontent.com/rhoai-genaiops/deploy-lab/main/university-data/canopy-in-botany.pdf",
]

# 수집할 대상 표시
print("📖 Ingesting documents into RAG system...")
for i, url in enumerate(urls):
    print(f"  • Document {i+1}: {url}")

# 새 API를 사용해 파일 다운로드 및 업로드
import requests
from io import BytesIO

try:
    uploaded_file_ids = []
    
    for i, url in enumerate(urls):
        print(f"\n📥 Downloading document from: {url}")
        response = requests.get(url)
        response.raise_for_status()
        
        # 다운로드한 콘텐츠로 파일 형태의 객체 생성
        file_content = BytesIO(response.content)
        file_content.name = f"canopy-in-botany-{i}.pdf"
        
        # 새 files API로 파일 업로드
        uploaded_file = client.files.create(
            file=file_content,
            purpose="assistants"  # 필수 purpose 파라미터
        )
        
        uploaded_file_ids.append(uploaded_file.id)
        print(f"✅ Uploaded file with ID: {uploaded_file.id}")
    
    # 청크 분할 설정과 함께 파일을 vector store에 추가
    for file_id in uploaded_file_ids:
        client.vector_stores.files.create(
            vector_store_id=vs.id,
            file_id=file_id,
            chunking_strategy={
                "type": "static",
                "static": {
                    "max_chunk_size_tokens": 512,
                    "chunk_overlap_tokens": 50
                }
            }
        )
        print(f"✅ Added file {file_id} to vector store with chunking")
    
    print("\n✅ Document ingestion complete!")
    print("🎯 Your documents are now searchable via semantic similarity!")
    
except Exception as e:
    print(f"\n❌ Document ingestion failed: {e}")
    print("💡 This might be due to PDF processing issues or network connectivity. Try with different documents or check the PDF accessibility.")

In [ ]:
queries = [
    "What are the types of Canopy?",
    "What is the structure of Canopy?",
]

## 먼저 RAG 없이 해 보기
비교 대상을 만들기 위해 먼저 RAG 없이 응답을 테스트해 보겠습니다.  
시스템 프롬프트를 포함해, 앞에서 모델에 프롬프트를 보낼 때 사용한 코드와 매우 비슷하다는 점에 주목하세요.

In [ ]:
for prompt in queries:
    cprint(f"\nUser> {prompt}", "blue")

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        stream=True,
    )

    cprint("inference> ", color="magenta", end='')
    
    # 스트리밍 응답 처리 - 토큰이 하나씩 도착함
    for chunk in response:
        if hasattr(chunk, 'choices') and chunk.choices:
            delta = chunk.choices[0].delta
            if hasattr(delta, 'content') and delta.content:
                cprint(delta.content, color="magenta", end='')
        elif hasattr(chunk, 'content'):
            cprint(chunk.content, color="magenta", end='')

## 🔍 RAG 검색과 생성 테스트하기

이제 전체 **RAG 파이프라인**을 테스트해 보겠습니다.

In [ ]:
for prompt in queries:
    cprint(f"\nUser> {prompt}", "blue")
    
    # 벡터 데이터베이스에 질의해 관련 문서 청크 찾기
    search_results = client.vector_stores.search(
        vector_store_id=vs.id,
        query=prompt,
        max_num_results=5,
        search_mode="vector"  # 벡터 유사도 검색 사용
    )

    retrieved_chunks = []
    for i, result in enumerate(search_results.data):
        chunk_content = result.content if hasattr(result, 'content') else str(result)
        metadata = result.metadata if hasattr(result, 'metadata') else {}
        retrieved_chunks.append(f"Result {i+1}\nContent: {chunk_content}\nMetadata: {metadata}")
    
    rag_response_content = "\n\n".join(retrieved_chunks)
    
    cprint(f"Text chunks from vector search found: {len(search_results.data)} chunks")
    cprint(f"\n--- Search Results ---", "yellow")
    cprint(rag_response_content[:500] + "..." if len(rag_response_content) > 500 else rag_response_content, "cyan")

    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]

    # 이제 검색된 콘텐츠를 컨텍스트로 프롬프트에 주입합니다. LLM이 문서 정보를 얻는 곳이므로 RAG가 작동하게 하는 핵심 부분입니다
    extended_prompt = f"Please answer the given query using the context below.\n\nCONTEXT:\n{rag_response_content}\n\nQUERY:\n{prompt}"
    messages.append({"role": "user", "content": extended_prompt})

    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        stream=True,
    )

    cprint("inference> ", color="magenta", end='')
    
    # 스트리밍 응답 처리 - 토큰이 하나씩 도착함
    for chunk in response:
        if hasattr(chunk, 'choices') and chunk.choices:
            delta = chunk.choices[0].delta
            if hasattr(delta, 'content') and delta.content:
                cprint(delta.content, color="magenta", end='')
        elif hasattr(chunk, 'content'):
            cprint(chunk.content, color="magenta", end='')

## 🎉 OGX로 완전한 RAG 시스템을 만들었습니다!

그런데... 프로덕션 수준의 시스템이 되려면 아직 갈 길이 남아 있습니다! 예를 들어, 벡터 데이터베이스에 새 문서를 넣을 때마다 노트북을 실행하라고 할 수는 없겠죠? 

이제 자동화된 데이터 수집 파이프라인의 세계로 들어가 보겠습니다!